# 🎯 YOLOX-S — Détection de cibles radar

Ce notebook permet de lancer l'entraînement du modèle YOLOX-S sur le dataset SOC_50classes depuis Google Colab (GPU T4).

**Étapes :**
1. Vérification du GPU
2. Installation des dépendances
3. Upload et extraction du dataset
4. Clone du projet et configuration
5. Lancement de l'entraînement
6. Visualisation des métriques (MLflow via ngrok)
7. Téléchargement des checkpoints

## 1. Vérification du GPU

In [ ]:
import torch

print(f'GPU disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go')
else:
    print('⚠️  Pas de GPU détecté. Aller dans Exécution > Modifier le type d\'exécution > GPU')

## 2. Installation des dépendances

In [ ]:
# Installation de YOLOX (librairie officielle Megvii)
!pip install -q git+https://github.com/Megvii-BaseDetection/YOLOX.git

# Dépendances complémentaires
!pip install -q mlflow pycocotools opencv-python-headless tifffile Pillow

# ngrok pour exposer l'UI MLflow
!pip install -q pyngrok

print('✅ Installation terminée')

## 3. Upload et extraction du dataset

Préparez votre dataset en local sous forme d'archive `.zip` avec la structure suivante :
```
SOC_50classes_coco.zip
└── SOC_50classes_coco/
    ├── annotations/
    │   ├── train.json
    │   └── test.json
    └── images/
        ├── train/
        └── test/
```

In [ ]:
import os
from google.colab import files

# Option A : Upload manuel depuis votre machine
print('Uploadez votre archive dataset (.zip)...')
uploaded = files.upload()

In [ ]:
# Option B : Télécharger depuis Google Drive (décommenter si préféré)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/SOC_50classes_coco.zip /content/

# Extraction
import zipfile

zip_name = list(uploaded.keys())[0]  # Nom du fichier uploadé
print(f'Extraction de : {zip_name}')

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

# Vérification
!ls /content/SOC_50classes_coco/
!ls /content/SOC_50classes_coco/annotations/
print('✅ Dataset extrait')

## 4. Clone du projet et configuration

In [ ]:
import os

# ── À MODIFIER : URL de votre dépôt Git ──────────────────────────────────
REPO_URL = 'https://github.com/william94000schr/ATR_SAR.git'  # <-- modifier
PROJECT_DIR = '/content/radar_detection'
# ─────────────────────────────────────────────────────────────────────────

!git clone {REPO_URL} {PROJECT_DIR}
os.chdir(PROJECT_DIR)
print(f'Répertoire de travail : {os.getcwd()}')
!ls

In [ ]:
# Création du lien symbolique vers le dataset
!mkdir -p data
!ln -sf /content/SOC_50classes_coco data/SOC_50classes_coco

# Vérification de la structure
!ls data/SOC_50classes_coco/
!echo "Train images :" $(ls data/SOC_50classes_coco/images/train | wc -l)
!echo "Test  images :" $(ls data/SOC_50classes_coco/images/test  | wc -l)
print('✅ Dataset lié')

In [ ]:
# Vérification rapide d'une image
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

sample = next(Path('data/SOC_50classes_coco/images/train').glob('*.tif'))
img = Image.open(sample)
arr = np.array(img)
print(f'Fichier : {sample.name}')
print(f'Taille : {img.size}, Mode : {img.mode}, dtype : {arr.dtype}')
print(f'Min/Max : {arr.min()} / {arr.max()}')

plt.figure(figsize=(4, 4))
plt.imshow(arr, cmap='gray')
plt.title(sample.name)
plt.axis('off')
plt.show()

## 5. Lancement de l'entraînement

In [ ]:
# Paramètres d'entraînement
MAX_EPOCH = 100         # Réduire à 30-50 pour un test rapide
BATCH_SIZE = 32         # Adapter selon la VRAM disponible
RUN_NAME = 'yolox_s_colab_v1'
FP16 = True             # Mixed precision : économise la VRAM sur T4

print(f'Config : {MAX_EPOCH} epochs, batch={BATCH_SIZE}, fp16={FP16}')

In [ ]:
# Lancement
!python train.py \
    --config config/yolox_s_radar.py \
    --max-epoch {MAX_EPOCH} \
    --batch-size {BATCH_SIZE} \
    --run-name {RUN_NAME} \
    {'--fp16' if FP16 else '--no-fp16'}

## 6. Visualisation MLflow via ngrok

MLflow UI est exposé publiquement via ngrok le temps de la session.

In [ ]:
from pyngrok import ngrok
import subprocess
import time

# ── À MODIFIER : token ngrok gratuit sur https://dashboard.ngrok.com ──────
NGROK_TOKEN = 'VOTRE_TOKEN_NGROK'  # <-- modifier
# ─────────────────────────────────────────────────────────────────────────

ngrok.set_auth_token(NGROK_TOKEN)

# Démarrage du serveur MLflow en arrière-plan
mlflow_proc = subprocess.Popen(
    ['mlflow', 'ui', '--backend-store-uri', 'outputs/mlruns', '--port', '5000'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)  # Attendre le démarrage

# Tunnel ngrok
tunnel = ngrok.connect(5000)
print(f'✅ MLflow UI accessible : {tunnel.public_url}')

## 7. Évaluation sur le test set

In [ ]:
# Chercher le meilleur checkpoint
import glob

checkpoints = glob.glob('outputs/checkpoints/**/best_ckpt.pth', recursive=True)
if checkpoints:
    best_ckpt = checkpoints[0]
    print(f'Meilleur checkpoint : {best_ckpt}')
else:
    # Fallback sur le dernier checkpoint
    checkpoints = glob.glob('outputs/checkpoints/**/*.pth', recursive=True)
    best_ckpt = sorted(checkpoints)[-1] if checkpoints else None
    print(f'Checkpoint utilisé : {best_ckpt}')

In [ ]:
if best_ckpt:
    !python evaluate.py \
        --config config/yolox_s_radar.py \
        --ckpt {best_ckpt} \
        --conf 0.01
else:
    print('⚠️  Aucun checkpoint trouvé. Vérifier que l\'entraînement s\'est terminé.')

## 8. Téléchargement des résultats

In [ ]:
import shutil
from google.colab import files

# Archive des checkpoints + logs MLflow
shutil.make_archive('/content/results', 'zip', 'outputs')
files.download('/content/results.zip')
print('✅ Archive téléchargée : results.zip')